# 1. Setup: Packages and Global Parameters


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import csv
import glob
import json
import os

# Folder in Google Drive containing the batch JSON files to combine.
BATCH_DIR = "/content/drive/MyDrive/phd/P4/Data"
BATCH_PATTERN = "batch*.json"
OUTPUT_CSV = os.path.join(BATCH_DIR, "aiid_full_incidents_processed.csv")

FIELDNAMES = [
    "incident_id", "title", "description", "date", "date_modified", "year",
    "developer", "deployer", "harmed", "implicated_systems", "report_count",
    "mit_risk_domain", "mit_risk_subdomain", "mit_entity", "mit_intent", "mit_timing",
    "cset_sector", "cset_lives_lost", "cset_injuries", "cset_location_country",
    "cset_ai_system_description", "cset_entities_raw",
    "gmf_known_ai_goal", "gmf_known_technology", "gmf_known_technical_failure",
    "n_cset_annotators",
]


def names_join(items):
    return "; ".join(x.get("name", "") for x in (items or []) if x)


def find_classification(classifications, namespace, require_publish=False):
    for c in classifications or []:
        if c.get("namespace") == namespace and (not require_publish or c.get("publish")):
            return c
    return None


def count_namespace_prefix(classifications, prefix):
    return sum(1 for c in classifications or [] if str(c.get("namespace", "")).startswith(prefix))


def classification_attr(classification, short_name):
    """Look up an attribute by short_name and decode its JSON-encoded value_json."""
    if not classification:
        return None
    for a in classification.get("attributes", []):
        if a.get("short_name") == short_name:
            try:
                return json.loads(a["value_json"])
            except (TypeError, ValueError):
                return a["value_json"]
    return None


def build_row(incident):
    classifications = incident.get("classifications") or []
    mit = find_classification(classifications, "MIT")
    csetv1 = find_classification(classifications, "CSETv1")
    # GMF ("known AI goal/technology/failure") is only trustworthy once published.
    gmf = find_classification(classifications, "GMF", require_publish=True)

    date = incident.get("date") or ""
    sector = classification_attr(csetv1, "Sector of Deployment")
    lives_lost = classification_attr(csetv1, "Lives Lost")
    injuries = classification_attr(csetv1, "Injuries")

    return {
        "incident_id": incident.get("incident_id"),
        "title": incident.get("title"),
        "description": incident.get("description"),
        "date": date,
        "date_modified": incident.get("date_modified") or "",
        "year": date[:4] if date else "",
        "developer": names_join(incident.get("AllegedDeveloperOfAISystem")),
        "deployer": names_join(incident.get("AllegedDeployerOfAISystem")),
        "harmed": names_join(incident.get("AllegedHarmedOrNearlyHarmedParties")),
        "implicated_systems": names_join(incident.get("implicated_systems")),
        "report_count": len(incident.get("reports") or []),
        "mit_risk_domain": classification_attr(mit, "Risk Domain") or "",
        "mit_risk_subdomain": classification_attr(mit, "Risk Subdomain") or "",
        "mit_entity": classification_attr(mit, "Entity") or "",
        "mit_intent": classification_attr(mit, "Intent") or "",
        "mit_timing": classification_attr(mit, "Timing") or "",
        "cset_sector": "" if sector is None else str(sector),
        "cset_lives_lost": float(lives_lost) if isinstance(lives_lost, (int, float)) else "",
        "cset_injuries": float(injuries) if isinstance(injuries, (int, float)) else "",
        "cset_location_country": classification_attr(csetv1, "Location Country (two letters)") or "",
        "cset_ai_system_description": classification_attr(csetv1, "AI System Description") or "",
        # csetv1 present but attribute missing/null -> "null"; csetv1 absent entirely -> "".
        "cset_entities_raw": json.dumps(classification_attr(csetv1, "Entities")) if csetv1 is not None else "",
        "gmf_known_ai_goal": json.dumps(classification_attr(gmf, "Known AI Goal")) if gmf is not None else "",
        "gmf_known_technology": json.dumps(classification_attr(gmf, "Known AI Technology")) if gmf is not None else "",
        "gmf_known_technical_failure": json.dumps(classification_attr(gmf, "Known AI Technical Failure")) if gmf is not None else "",
        # Count of CSETv1_Annotator-* classification blocks, published or not.
        "n_cset_annotators": count_namespace_prefix(classifications, "CSETv1_Annotator-"),
    }


batch_paths = sorted(glob.glob(os.path.join(BATCH_DIR, BATCH_PATTERN)))
print(f"Found {len(batch_paths)} batch file(s)")

incidents_by_id = {}

for path in batch_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Each batch file is a raw GraphQL response: {"data": {"incidents": [...]}}.
    records = next(iter(data["data"].values())) if isinstance(data, dict) and "data" in data else data

    for incident in records:
        incidents_by_id[incident["incident_id"]] = incident

    print(f"  {os.path.basename(path)}: {len(records)} record(s)")

print(f"Total unique incidents: {len(incidents_by_id)}")

rows = [build_row(inc) for inc in incidents_by_id.values()]
rows.sort(key=lambda r: r["incident_id"])

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {OUTPUT_CSV}")

# 2. Descriptive Analysis


In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/phd/P4/Data/aiid_full_incidents_processed.csv"

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("Columns and dtypes:")
print(df.dtypes)

In [ ]:
# Missing values per column
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0]

In [ ]:
# Summary statistics for numeric columns
df.describe(include="number").T

In [ ]:
# Value counts for object/categorical columns with manageable cardinality
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    n_unique = df[col].nunique(dropna=True)
    if 1 < n_unique <= 30:
        print(f"--- {col} ({n_unique} unique values) ---")
        print(df[col].value_counts(dropna=False).head(10))
        print()

In [ ]:
# Incidents over time, if a date-like column is present (e.g. "date")
date_cols = [c for c in df.columns if "date" in c.lower()]

if date_cols:
    date_col = date_cols[0]
    dates = pd.to_datetime(df[date_col], errors="coerce")
    print(f"Using date column: {date_col}")
    print(f"Range: {dates.min()} to {dates.max()}")
    print()
    print("Incidents per year:")
    print(dates.dt.year.value_counts().sort_index())
else:
    print("No date-like column found.")

# 3. Module 1 --- Compositional Dynamics (H1, H1a)

Tests whether the composition of realized AI incidents shifted materially
over the primary analysis window (2018-2025), and whether that shift
reflects displacement of established risk-domain categories or the
emergence of new ones alongside a stable base (H1a).


In [ ]:
import re

PRIMARY_WINDOW = (2018, 2025)
BREAK_YEAR = 2022  # pre = up to and including 2021; post = 2022 onward

df["year"] = pd.to_numeric(df["year"], errors="coerce")

primary = df[df["year"].between(*PRIMARY_WINDOW)].copy()
classified = primary[primary["mit_risk_domain"].notna() & (primary["mit_risk_domain"] != "")].copy()

# MIT risk domains are prefixed "N. <name>" (e.g. "4. Malicious Actors and Misuse");
# sort/display by that leading number rather than alphabetically.
def domain_sort_key(domain):
    m = re.match(r"\s*(\d+)", str(domain))
    return int(m.group(1)) if m else 999

domains = sorted(classified["mit_risk_domain"].unique(), key=domain_sort_key)

n_primary = len(primary)
n_classified = len(classified)
print(f"Primary window {PRIMARY_WINDOW[0]}-{PRIMARY_WINDOW[1]}: n = {n_primary}")
print(f"Classified under MIT taxonomy: n = {n_classified} ({n_classified / n_primary:.1%})")
print()
print("Risk domains:")
for d in domains:
    print(" ", d)

In [ ]:
# Table 2: annual risk-domain composition (% of classified incidents per year)
year_domain_counts = (
    classified.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)[domains]
)
year_totals = year_domain_counts.sum(axis=1)
year_domain_pct = year_domain_counts.div(year_totals, axis=0) * 100

table2 = year_domain_pct.round(1).copy()
table2["n"] = year_totals
table2

In [ ]:
# Table 3: absolute incident counts, pre- vs post-BREAK_YEAR
classified["period"] = classified["year"].apply(
    lambda y: f"Pre-{BREAK_YEAR}" if y < BREAK_YEAR else f"Post-{BREAK_YEAR}"
)

period_counts = classified.groupby(["mit_risk_domain", "period"]).size().unstack(fill_value=0)
pre_col, post_col = f"Pre-{BREAK_YEAR}", f"Post-{BREAK_YEAR}"
period_counts = period_counts.reindex(columns=[pre_col, post_col], fill_value=0).reindex(domains)

pre_total, post_total = period_counts[pre_col].sum(), period_counts[post_col].sum()
pre_pct = period_counts[pre_col] / pre_total * 100
post_pct = period_counts[post_col] / post_total * 100

table3 = pd.DataFrame({
    f"{pre_col} (n={pre_total})": period_counts[pre_col].astype(str) + " (" + pre_pct.round(1).astype(str) + "%)",
    f"{post_col} (n={post_total})": period_counts[post_col].astype(str) + " (" + post_pct.round(1).astype(str) + "%)",
    "Change (count)": period_counts[post_col] - period_counts[pre_col],
    "Change (share, pp)": (post_pct - pre_pct).round(1),
}).sort_values("Change (count)", ascending=False)

table3

In [ ]:
%pip install -q ruptures

import numpy as np
import ruptures as rpt

# Structural break test on the Malicious Actors & Misuse share series.
mam_domain = next(d for d in domains if "malicious actors" in d.lower())
mam_share = year_domain_pct[mam_domain].sort_index()
years = mam_share.index.to_numpy()
signal = mam_share.to_numpy()

# PELT selects its own number of breakpoints via a penalty (higher = fewer
# breaks); binary segmentation is asked directly for a single breakpoint.
detectors = [
    ("PELT", rpt.Pelt(model="l2").fit(signal), {"pen": 3}),
    ("Binary segmentation", rpt.Binseg(model="l2").fit(signal), {"n_bkps": 1}),
]

for algo_name, algo, kwargs in detectors:
    # predict() returns breakpoint indices, the last of which is len(signal) (end marker).
    breakpoints = [b for b in algo.predict(**kwargs) if b < len(signal)]
    break_years = [int(years[b]) for b in breakpoints]
    print(f"{algo_name}: break(s) at start of {break_years}" if break_years else f"{algo_name}: no break detected")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# Poisson regression of annual incident count on year, risk domain, and
# their interaction. A significant interaction (via LR test against the
# no-interaction model) rejects parallel category-specific trends.
year_domain_long = (
    classified.groupby(["year", "mit_risk_domain"]).size().reset_index(name="count")
)
year_domain_long["mit_risk_domain"] = pd.Categorical(
    year_domain_long["mit_risk_domain"], categories=domains
)

full_model = smf.glm(
    "count ~ C(year) * C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()
reduced_model = smf.glm(
    "count ~ C(year) + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()

lr_stat = 2 * (full_model.llf - reduced_model.llf)
lr_df = full_model.df_model - reduced_model.df_model
lr_pvalue = chi2.sf(lr_stat, lr_df)

print(f"LR test for parallel trends: LR = {lr_stat:.1f}, df = {lr_df:.0f}, p = {lr_pvalue:.3g}")
print()

# Per-category annual growth rate: exp(year coefficient) from a separate
# log-linear Poisson fit of count ~ year within each domain.
print("Estimated annual growth multiplier by risk domain:")
for d in domains:
    sub = year_domain_long[year_domain_long["mit_risk_domain"] == d]
    m = smf.glm("count ~ year", data=sub, family=sm.families.Poisson()).fit()
    growth = np.exp(m.params["year"])
    print(f"  {d}: {growth:.3f} ({(growth - 1):+.1%} per year)")

### Linear-trend robustness specification (primary reported statistic)

The interaction test above uses a fully saturated `C(year) * C(mit_risk_domain)`
model, which throws a `PerfectSeparationWarning` from zero-count year-domain
cells and has df = 39. The paper instead reports a more parsimonious linear
year-trend interaction test (`t * C(mit_risk_domain)`, df = 6) as the primary
statistic. This cell adds that specification without altering the saturated
model above.

In [ ]:
import warnings

# Reuse year_domain_long from the cell above; add a numeric time index if
# not already present (t = year - year.min(), i.e. 0-7 for 2018-2025).
if "t" not in year_domain_long.columns:
    year_domain_long["t"] = year_domain_long["year"] - year_domain_long["year"].min()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    full_linear = smf.glm(
        "count ~ t * C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
    ).fit()
    reduced_linear = smf.glm(
        "count ~ t + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
    ).fit()

lr_linear = 2 * (full_linear.llf - reduced_linear.llf)
lr_linear_df = full_linear.df_model - reduced_linear.df_model
lr_linear_p = chi2.sf(lr_linear, lr_linear_df)

print(f"LR test for parallel trends (linear year trend): LR = {lr_linear:.1f}, "
      f"df = {lr_linear_df:.0f}, p = {lr_linear_p:.3g}")

separation_warnings = [w for w in caught if "eparation" in str(w.message)]
if separation_warnings:
    print("\nWARNING: PerfectSeparationWarning raised for the linear-trend model "
          "(unexpected -- investigate before reporting this statistic):")
    for w in separation_warnings:
        print(" ", w.category.__name__, "-", w.message)
else:
    print("No PerfectSeparationWarning raised for the linear-trend model (as expected).")

# Per-domain growth rate under the linear (t) spec, cross-checked against the
# `count ~ year` spec from the previous cell. exp(coef on `year`) and
# exp(coef on `t = year - min(year)`) should be numerically identical --
# only the intercept absorbs the shift -- so confirm rather than assume.
print("\nPer-domain growth multiplier, t-spec vs. year-spec:")
mismatches = []
for d in domains:
    sub = year_domain_long[year_domain_long["mit_risk_domain"] == d]
    m_year = smf.glm("count ~ year", data=sub, family=sm.families.Poisson()).fit()
    m_t = smf.glm("count ~ t", data=sub, family=sm.families.Poisson()).fit()
    growth_year = np.exp(m_year.params["year"])
    growth_t = np.exp(m_t.params["t"])
    match = np.isclose(growth_year, growth_t, rtol=1e-6)
    if not match:
        mismatches.append((d, growth_year, growth_t))
    print(f"  {d}: t-spec = {growth_t:.3f}, year-spec = {growth_year:.3f} {'OK' if match else 'MISMATCH'}")

if mismatches:
    print("\nWARNING: growth multipliers differ between the `t` and `year` specifications:")
    for d, gy, gt in mismatches:
        print(f"  {d}: year-spec={gy:.6f} vs t-spec={gt:.6f}")
else:
    print("\nConfirmed: growth multipliers are identical (as expected) between specs.")

In [ ]:
from scipy.stats import norm


def mann_kendall(series):
    """Original (non-seasonal) Mann-Kendall trend test.

    Returns (trend, S, p_value) where trend is 'increasing', 'decreasing',
    or 'no trend' at the 0.05 level.
    """
    x = np.asarray(series, dtype=float)
    n = len(x)
    s = sum(np.sign(x[j] - x[i]) for i in range(n - 1) for j in range(i + 1, n))

    # Variance of S (no tie correction needed: annual incident counts are
    # essentially continuous-valued for n >= ~8 non-degenerate years).
    var_s = n * (n - 1) * (2 * n + 5) / 18

    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0

    p_value = 2 * (1 - norm.cdf(abs(z)))
    if p_value < 0.05:
        trend = "increasing" if s > 0 else "decreasing"
    else:
        trend = "no trend"
    return trend, s, p_value


# Trend test on each domain's full annual absolute-count series (2018-2025).
annual_counts = classified.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)
annual_counts = annual_counts.reindex(
    index=range(PRIMARY_WINDOW[0], PRIMARY_WINDOW[1] + 1), columns=domains, fill_value=0
)

print("Mann-Kendall trend test (annual absolute incident count, 2018-2025):")
for d in domains:
    trend, s, p = mann_kendall(annual_counts[d])
    print(f"  {d}: {trend} (S = {s:.0f}, p = {p:.3g})")

Note: three of seven domains show a statistically significant trend at the 5%
level over 2018-2025 -- Malicious Actors & Misuse, Misinformation, and
Privacy & Security are all increasing; the remaining four (Discrimination
and Toxicity, AI System Safety and Failures, Human-Computer Interaction,
Socioeconomic and Environmental Harms) show no significant trend. This is
inconsistent with a paper draft that describes only two categories as
increasing, and should be reconciled before finalizing Section 4.1.

## Window Robustness: Sensitivity to Analysis Start Year

Re-runs the linear year-trend interaction test and the Malicious Actors &
Misuse Mann-Kendall trend test across three window start years (2016, 2018,
2020), all ending in 2025, to check whether the compositional finding is
sensitive to where the primary analysis window begins.

In [ ]:
START_YEARS = [2016, 2018, 2020]
END_YEAR = 2025

window_rows = []
mk_by_window = {}
mk_p_by_window = {}

for start_year in START_YEARS:
    window = df[
        df["year"].between(start_year, END_YEAR)
        & df["mit_risk_domain"].notna()
        & (df["mit_risk_domain"] != "")
        & df["mit_risk_domain"].isin(domains)
    ].copy()

    long_w = window.groupby(["year", "mit_risk_domain"]).size().reset_index(name="count")
    long_w["mit_risk_domain"] = pd.Categorical(long_w["mit_risk_domain"], categories=domains)
    long_w["t"] = long_w["year"] - start_year

    # Wide year x domain count grid for this window. Built via
    # size().unstack(fill_value=0) -- NOT pivot()+reindex(fill_value=0) --
    # because reindex's fill_value only fills newly introduced index/column
    # labels; it does not backfill the NaNs pivot() leaves for zero-count
    # cells that fall within an already-present year/domain. unstack's
    # fill_value correctly fills those reshape-introduced NaNs.
    counts_wide = (
        window.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)
        .reindex(index=range(start_year, END_YEAR + 1), columns=domains, fill_value=0)
    )

    zero_cells = [(d, int(y)) for y in counts_wide.index for d in domains if counts_wide.loc[y, d] == 0]
    if zero_cells:
        print(f"[start_year={start_year}] zero-count year/domain cell(s) (n={len(zero_cells)}):")
        for d, y in zero_cells:
            print(f"    {y} - {d}")

    full_w = smf.glm(
        "count ~ t * C(mit_risk_domain)", data=long_w, family=sm.families.Poisson()
    ).fit()
    red_w = smf.glm(
        "count ~ t + C(mit_risk_domain)", data=long_w, family=sm.families.Poisson()
    ).fit()
    lr_w = 2 * (full_w.llf - red_w.llf)
    lr_w_df = full_w.df_model - red_w.df_model
    lr_w_p = chi2.sf(lr_w, lr_w_df)

    window_rows.append({
        "start_year": start_year, "end_year": END_YEAR,
        "n": len(window), "LR": round(lr_w, 1), "df": int(lr_w_df), "p_value": lr_w_p,
    })

    # Mann-Kendall on the Malicious Actors & Misuse annual absolute count
    # for this window (reuses mann_kendall() defined in the cell above).
    # Both the trend label and its p-value are cached per window (keyed by
    # start_year) so downstream cells -- e.g. the Figure 1 plot -- can pull
    # them live instead of recomputing or, worse, hardcoding them.
    mam_series = counts_wide[mam_domain]
    trend, s, p = mann_kendall(mam_series)
    mk_by_window[start_year] = trend
    mk_p_by_window[start_year] = p
    print(f"[start_year={start_year}] Malicious Actors & Misuse Mann-Kendall: "
          f"{trend} (S={s:.0f}, p={p:.3g})")

window_robustness = pd.DataFrame(window_rows)
print()
print(window_robustness.to_string(index=False))

stable = len(set(mk_by_window.values())) == 1 and next(iter(mk_by_window.values())) == "increasing"
print()
if stable:
    print("Malicious Actors & Misuse trend classification is stable "
          "('increasing') across all three start years.")
else:
    print(f"Malicious Actors & Misuse trend classification is NOT stable "
          f"across start years: {mk_by_window}")

In [ ]:
import os

import matplotlib.pyplot as plt

# Figure 1: annual share of Malicious Actors & Misuse under each window
# specification, against the seven other risk domains shown faintly for
# reference. Every number that appears on the figure (n, LR test p-value,
# MK trend/p-value) is pulled live from window_robustness / mk_by_window /
# mk_p_by_window -- none of it is hardcoded, so the figure cannot drift out
# of sync with a later re-run of the underlying models.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

window_shares = {}
max_share_seen = 0.0
mam_handle = None

for start_year, ax in zip(START_YEARS, axes):
    window = df[
        df["year"].between(start_year, END_YEAR)
        & df["mit_risk_domain"].notna()
        & (df["mit_risk_domain"] != "")
        & df["mit_risk_domain"].isin(domains)
    ]

    # Per-year share of THIS window's own classified total -- not sliced
    # from the primary-window Table 2 percentages -- so shares sum to 100%
    # within each year of this window and pre-2018 years (start_year=2016)
    # are covered too.
    counts = (
        window.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)
        .reindex(index=range(start_year, END_YEAR + 1), columns=domains, fill_value=0)
    )
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    window_shares[start_year] = pct
    max_share_seen = max(max_share_seen, pct.to_numpy().max())

    for d in domains:
        if d == mam_domain:
            continue
        ax.plot(pct.index, pct[d], color="lightgray", linewidth=1, zorder=1)

    line, = ax.plot(
        pct.index, pct[mam_domain], color="#1f77b4", linewidth=2.5, marker="o",
        zorder=3, label="Malicious Actors & Misuse",
    )
    mam_handle = line

    row = window_robustness.loc[window_robustness["start_year"] == start_year].iloc[0]
    trend = mk_by_window[start_year]
    mk_p = mk_p_by_window[start_year]

    # row['n'] would print as e.g. "1218.0" here: extracting a single row
    # from a DataFrame with mixed int/float columns upcasts the whole row
    # to float64. Cast back to int so it matches the verification printout
    # below exactly (same significant figures, same rounding).
    ax.set_title(
        f"Window {start_year}-{END_YEAR} (n={int(row['n'])})\n"
        f"Poisson interaction p={row['p_value']:.1e}\n"
        f"MK trend: {trend} (p={mk_p:.3f})"
    )
    ax.set_xlabel("Year")

axes[0].set_ylabel("Share of classified incidents (%)")
axes[0].set_ylim(0, max_share_seen * 1.1)

fig.legend(handles=[mam_handle], labels=["Malicious Actors & Misuse"], loc="upper left")
fig.suptitle("Window robustness check: 'Malicious Actors & Misuse' share under alternative start years")
plt.tight_layout(rect=[0, 0, 1, 0.93])

figure1_path = os.path.join(BATCH_DIR, "figure1_window_robustness.png")
plt.savefig(figure1_path, dpi=300)
plt.show()
print(f"Saved {figure1_path}")

# Verification: the exact numbers that appear in the three panel titles,
# printed here so they can be eyeballed against the figure directly.
print()
print("Panel title values (cross-check against the figure):")
title_check = window_robustness.copy()
title_check["mk_trend"] = title_check["start_year"].map(mk_by_window)
title_check["mk_p"] = title_check["start_year"].map(mk_p_by_window)
for _, r in title_check.iterrows():
    print(f"  start_year={r['start_year']}: n={int(r['n'])}, "
          f"Poisson interaction p={r['p_value']:.1e}, "
          f"MK trend={r['mk_trend']} (p={r['mk_p']:.3f})")

## 3.5 Media-Volume Normalization Robustness Check (H1a)

Tests whether the H1a absolute-count findings survive normalization against
an index of AI-related media/news volume, addressing the objection that
rising AIID incident counts reflect rising media attention to AI rather than
rising underlying incidence.

In [ ]:
import io

import requests
from requests.adapters import HTTPAdapter, Retry

GDELT_QUERY = '("artificial intelligence" OR "machine learning")'


def fetch_gdelt_annual(query=GDELT_QUERY, y0=PRIMARY_WINDOW[0], y1=PRIMARY_WINDOW[1]):
    """Annual mean of GDELT DOC 2.0 daily volume intensity.

    'timelinevol' returns volume as a PERCENTAGE of all monitored articles,
    so it is already normalized for growth in total news output -- it
    isolates AI's *share* of media attention rather than conflating it with
    the general expansion of GDELT's crawl.
    """
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {
        "query": query,
        "mode": "timelinevol",
        "startdatetime": f"{y0}0101000000",
        "enddatetime": f"{y1}1231235959",
        "format": "csv",
    }
    # GDELT can be slow/flaky from cloud IPs (Colab included); retry a few
    # times with backoff rather than hanging on one long-timeout attempt.
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=Retry(
        total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504]
    )))
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    d = pd.read_csv(io.StringIO(r.text))
    # Column names vary slightly across GDELT responses; take date + value.
    date_col = [c for c in d.columns if "date" in c.lower()][0]
    val_col = [c for c in d.columns if c != date_col][-1]
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col])
    d["year"] = d[date_col].dt.year
    return d.groupby("year")[val_col].mean().reindex(range(y0, y1 + 1))


# FALLBACK: if GDELT is unreachable (timeouts, rate limiting -- common from
# cloud/Colab IPs) or its schema shifts, paste an annual series here by hand
# (e.g. from the Stanford AI Index media-attention data) and set
# USE_FALLBACK = True. Values need only be on a consistent relative scale --
# the offset is scale-invariant up to an intercept shift.
USE_FALLBACK = False
FALLBACK_INDEX = {y: None for y in range(PRIMARY_WINDOW[0], PRIMARY_WINDOW[1] + 1)}

# NOTE: raise RuntimeError, not SystemExit, below -- SystemExit inside a
# Jupyter/Colab cell breaks IPython's own traceback formatter (an "Internal
# Python error in the inspect module" crash unrelated to the real cause).
if USE_FALLBACK:
    media = pd.Series(FALLBACK_INDEX, name="media").astype(float)
else:
    try:
        media = fetch_gdelt_annual()
        media.name = "media"
    except Exception as e:
        raise RuntimeError(
            f"GDELT fetch failed ({e}). Set USE_FALLBACK=True and paste an "
            "annual series into FALLBACK_INDEX."
        ) from e

if media.isna().any():
    raise RuntimeError(f"Index has missing years:\n{media}")

print("AI news-volume index (annual):")
print(media.round(4).to_string())
print()

# High correlation with calendar year is expected -- it is why the index
# enters as a fixed-coefficient offset rather than an estimated covariate.
r_yt = np.corrcoef(media.index.values, media.values)[0, 1]
print(f"corr(index, calendar year) = {r_yt:.3f}")
print("  -> high correlation is expected; it is the reason the index enters")
print("     as a fixed-coefficient offset rather than an estimated covariate.")

In [ ]:
# Rescale index to mean 1 so normalized counts stay in incident-like units
# and remain directly comparable to the raw counts.
m = media / media.mean()

media_comparison = pd.DataFrame({
    "raw": year_domain_counts.sum(axis=1),
    "index": m.round(3),
    "normalized": (year_domain_counts.sum(axis=1) / m).round(1),
})
media_comparison

In [ ]:
# Per-category trend (H1a): annual growth factor exp(beta), raw vs. media-
# normalized (offset Poisson: log(m) enters as a fixed-coefficient offset).
year_domain_long["t"] = year_domain_long["year"] - PRIMARY_WINDOW[0]
year_domain_long["log_m"] = np.log(year_domain_long["year"].map(m).to_numpy())

print(f"{'Risk domain':<50}{'raw':>10}{'normalized':>13}")
print("-" * 73)

media_trend_rows = []
for d in domains:
    sub = year_domain_long[year_domain_long["mit_risk_domain"] == d]
    raw = smf.glm("count ~ t", data=sub, family=sm.families.Poisson()).fit()
    adj = smf.glm("count ~ t", data=sub, family=sm.families.Poisson(), offset=sub["log_m"]).fit()

    def fmt(res):
        b = np.exp(res.params["t"])
        p = res.pvalues["t"]
        star = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
        return f"{b:.3f}{star}"

    print(f"{d[:48]:<50}{fmt(raw):>10}{fmt(adj):>13}")
    media_trend_rows.append({
        "domain": d,
        "growth_raw": np.exp(raw.params["t"]), "p_raw": raw.pvalues["t"],
        "growth_norm": np.exp(adj.params["t"]), "p_norm": adj.pvalues["t"],
    })

media_trend = pd.DataFrame(media_trend_rows)
print("\n*** p<.001  ** p<.01  * p<.05")
print("Growth factor > 1 = increasing. Normalization shrinks all categories")
print("toward 1 because the common media trend is divided out.")

In [ ]:
# Year x category interaction (H1), raw vs. media-normalized. Uses a
# continuous linear trend (t) rather than the saturated C(year) model above,
# since the offset comparison requires a smooth exposure/trend specification.
full_raw = smf.glm(
    "count ~ t*C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()
red_raw = smf.glm(
    "count ~ t + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()
lr_raw = 2 * (full_raw.llf - red_raw.llf)
lr_raw_df = full_raw.df_model - red_raw.df_model

full_adj = smf.glm(
    "count ~ t*C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson(),
    offset=year_domain_long["log_m"],
).fit()
red_adj = smf.glm(
    "count ~ t + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson(),
    offset=year_domain_long["log_m"],
).fit()
lr_adj = 2 * (full_adj.llf - red_adj.llf)
lr_adj_df = full_adj.df_model - red_adj.df_model

print(f"  raw:        LR = {lr_raw:7.1f}, df = {lr_raw_df:.0f}, p = {chi2.sf(lr_raw, lr_raw_df):.2e}")
print(f"  normalized: LR = {lr_adj:7.1f}, df = {lr_adj_df:.0f}, p = {chi2.sf(lr_adj, lr_adj_df):.2e}")
print("""
These are near-identical by construction. A year-level multiplicative
factor scales every category identically within a year, so it cannot
create or destroy DIFFERENTIAL trends across categories -- and shares are
exactly invariant to it. With year fixed effects the offset is absorbed
completely and the statistic is unchanged to machine precision.

Report this as an analytic point, not an empirical discovery: the
compositional finding (H1) is immune to reporting-intensity drift by
construction. The normalization check earns its place against H1a, where
the claim concerns absolute levels.
""")

In [ ]:
# Mann-Kendall on raw vs. media-normalized annual counts (direct H1a check).
# Reuses the mann_kendall() function defined above rather than a separate
# pymannkendall dependency -- same test (original, non-seasonal MK).
print(f"{'Risk domain':<50}{'raw':>18}{'normalized':>20}")
print("-" * 88)

media_mk_rows = []
for d in domains:
    raw_series = annual_counts[d].astype(float)
    norm_series = annual_counts[d] / m.reindex(annual_counts.index)

    trend_raw, _, p_raw = mann_kendall(raw_series)
    trend_norm, _, p_norm = mann_kendall(norm_series)

    print(f"{d[:48]:<50}{trend_raw + ' ' + format(p_raw, '.3f'):>18}"
          f"{trend_norm + ' ' + format(p_norm, '.3f'):>20}")
    media_mk_rows.append({
        "domain": d, "trend_raw": trend_raw, "p_raw": p_raw,
        "trend_norm": trend_norm, "p_norm": p_norm,
    })

media_mk = pd.DataFrame(media_mk_rows)

print("""
n = 8 annual observations, so Mann-Kendall power is low; treat a shift from
'increasing' to 'no trend' as weak evidence rather than a refutation.

INTERPRETATION. The normalized series is a CONSERVATIVE LOWER BOUND, not a
preferred estimate. Media coverage of AI harm rose partly *because* AI harm
rose, so dividing by coverage removes real signal alongside reporting bias
(a bad-control problem). Report raw and normalized side by side and say so
explicitly. If misuse still trends up after normalization, that is a strong
result. If it flattens, that is ambiguous -- not a refutation of H1a.
""")

In [ ]:
import os

media_check_output = media_trend.merge(media_mk, on="domain")
media_check_output.to_csv(os.path.join(BATCH_DIR, "module1_media_normalization.csv"), index=False)
media_comparison.to_csv(os.path.join(BATCH_DIR, "module1_media_index.csv"))
print(f"Written module1_media_normalization.csv and module1_media_index.csv to {BATCH_DIR}")

# 4. Module 2 --- Correspondence with Postulated Threat Catalogues (H2)

Loads the pre-computed per-threat classification crosswalk (Claude / GPT /
Manual coders against the 55-threat catalogue) and its aggregate per-domain
distributions, already produced outside this notebook. Copy the three
`module2_*.csv` files (see `data/` in the repo) into `BATCH_DIR` in Drive
before running this section.

In [ ]:
MODULE2_AGG_CSV = os.path.join(BATCH_DIR, "module2_aggregate_distributions.csv")
MODULE2_CROSSWALK_CSV = os.path.join(BATCH_DIR, "module2_crosswalk_raw_per_threat.csv")
MODULE2_PAIRWISE_CSV = os.path.join(BATCH_DIR, "module2_pairwise_statistics.csv")

module2_agg = pd.read_csv(MODULE2_AGG_CSV)
module2_crosswalk = pd.read_csv(MODULE2_CROSSWALK_CSV)
module2_pairwise = pd.read_csv(MODULE2_PAIRWISE_CSV)

print(f"Aggregate distributions: {module2_agg.shape[0]} domains")
print(module2_agg.to_string(index=False))
print()
print(f"Per-threat crosswalk: {module2_crosswalk.shape[0]} threats")
print()
print("Pairwise statistics (as previously reported):")
print(module2_pairwise.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

SHORT_LABEL = {
    "Malicious Actors & Misuse": "Misuse",
    "AI System Safety, Failures, and Limitations": "Safety",
    "Discrimination and Toxicity": "Discrim.",
    "Misinformation": "Misinfo",
    "Privacy & Security": "Privacy",
    "Human-Computer Interaction": "HCI",
    "Socioeconomic & Environmental Harms": "Socioec.",
}

coders = [("claude_pct", "Claude"), ("gpt_pct", "GPT"), ("manual_pct", "Manual")]

# Single shared axis limit computed ONCE, before the panel loop. With
# sharex=True/sharey=True, a per-panel set_xlim/set_ylim call silently
# overrides every other panel's limits (whichever panel's call runs last
# wins), which would invisibly clip points from panels with a smaller data
# range (e.g. Manual tops out near 49% while Claude/GPT top out near
# 64-66% for Privacy & Security) with no error or warning.
all_values = pd.concat([module2_agg[col] for col, _ in coders] + [module2_agg["realized_pct"]])
lim = 1.15 * all_values.max()

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

pearson_computed = {}

for (col, name), ax in zip(coders, axes):
    x = module2_agg[col].to_numpy()
    y = module2_agg["realized_pct"].to_numpy()
    r, _ = pearsonr(x, y)
    pearson_computed[name] = r

    ax.scatter(x, y, color="#2a78d6", s=90, edgecolor="white", zorder=3)
    for xi, yi, label in zip(x, y, module2_agg["mit_domain_label"]):
        ax.annotate(
            SHORT_LABEL.get(label, label), (xi, yi),
            textcoords="offset points", xytext=(6, 4), fontsize=9,
        )

    ax.plot([0, lim], [0, lim], color="lightgray", linestyle="--", linewidth=1, zorder=1)

    slope, intercept = np.polyfit(x, y, 1)
    fit_x = np.array([0, lim])
    ax.plot(fit_x, slope * fit_x + intercept, color="orange", linewidth=2, zorder=2)

    ax.set_title(f"{name}\nPearson r = {r:.3f}")
    ax.set_xlabel("Catalogue share (%)")
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)

axes[0].set_ylabel("Realized incident share (%)")

# Verify all 7 points are visible in all 3 panels before saving.
for col, name in coders:
    exceeds = ((module2_agg[col] > lim) | (module2_agg["realized_pct"] > lim)).any()
    if exceeds:
        raise RuntimeError(f"{name}: a point exceeds the shared axis limit ({lim:.1f}) -- fix before saving.")
print(f"Shared axis limit: {lim:.1f}. Confirmed all 7 points visible in all 3 panels.")

fig.suptitle(
    "Catalogue Share vs. Realized Share, by MIT Harm Domain\n"
    "(gray dashed = perfect correspondence; orange = actual fit)"
)
plt.tight_layout(rect=[0, 0, 1, 0.90])

figure_scatter_path = "/content/drive/MyDrive/phd/P4/Data/figure_module2_catalogue_vs_realized.png"
plt.savefig(figure_scatter_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {figure_scatter_path}")

# Verification: reload the saved PNG and check its dimensions, and compare
# the computed Pearson r values against those already reported in the
# paper draft (per module2_pairwise_statistics.csv) to confirm the
# figure's titles match.
from PIL import Image

with Image.open(figure_scatter_path) as img:
    print(f"\nSaved figure dimensions: {img.size[0]} x {img.size[1]} px")

print()
print("Pearson r: computed here vs. paper draft values")
paper_r = {"Claude": -0.129, "GPT": -0.120, "Manual": 0.046}
for name in ["Claude", "GPT", "Manual"]:
    computed = pearson_computed[name]
    reported = paper_r[name]
    match = "MATCH" if abs(computed - reported) < 0.001 else "DIFFERS"
    print(f"  {name}: computed = {computed:.3f}, paper draft = {reported:.3f} [{match}]")

In [ ]:
gap_df = module2_agg.copy()
gap_df["short_label"] = gap_df["mit_domain_label"].map(lambda l: SHORT_LABEL.get(l, l))
gap_df["gap_claude"] = gap_df["realized_pct"] - gap_df["claude_pct"]
gap_df["gap_gpt"] = gap_df["realized_pct"] - gap_df["gpt_pct"]
gap_df["gap_manual"] = gap_df["realized_pct"] - gap_df["manual_pct"]
gap_df["gap_avg"] = gap_df[["gap_claude", "gap_gpt", "gap_manual"]].mean(axis=1)

# Ascending sort + matplotlib's default barh ordering (index 0 at the
# bottom) reads exactly as required: most-overrepresented-in-catalogue
# (most negative) at the bottom, most-underrepresented (most positive) at
# the top.
gap_sorted = gap_df.sort_values("gap_avg", ascending=True).reset_index(drop=True)

GAP_COLORS = {"Claude": "#888780", "GPT": "#d3d1c7", "Manual": "#eda100"}

fig, ax = plt.subplots(figsize=(9, 6))

y_pos = np.arange(len(gap_sorted))
h = 0.25

ax.barh(y_pos - h, gap_sorted["gap_claude"], height=h, color=GAP_COLORS["Claude"], label="Claude")
ax.barh(y_pos, gap_sorted["gap_gpt"], height=h, color=GAP_COLORS["GPT"], label="GPT")
ax.barh(y_pos + h, gap_sorted["gap_manual"], height=h, color=GAP_COLORS["Manual"], label="Manual")

ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(gap_sorted["short_label"])
ax.set_xlabel("Realized % − Catalogue % (percentage points)")
ax.set_title(
    "Where the Catalogue Misses Reality, by MIT Harm Domain\n"
    "(positive = underrepresented in catalogue; negative = overrepresented)"
)
ax.legend()
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()

figure_gap_path = "/content/drive/MyDrive/phd/P4/Data/figure_module2_catalogue_gap_barchart.png"
plt.savefig(figure_gap_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {figure_gap_path}")

print()
print("Sorted gap table (percentage points, realized - catalogue):")
print(
    gap_sorted[["short_label", "gap_claude", "gap_gpt", "gap_manual"]]
    .round(1).to_string(index=False)
)

# 5. Module 3 --- Documented Legal and Financial Outcomes (Original Data)

Descriptive analysis of the hand-coded outcome sheet: for a stratified
sample of 150 incidents (drawn across risk domain and year), a systematic
search for documented legal or financial outcomes (litigation, settlements,
regulatory enforcement, quantified fines). Every number below is cross-checked
against the paper draft's Section 4.3 in Step 3.

In [ ]:
# Step 1: Load and validate
MODULE3_CSV = os.path.join(BATCH_DIR, "module3_coding_outcome_sheet.csv")

# Hand-coded outcome sheet, not the main AIID export -- read with
# dtype=str, keep_default_na=False (this project's established convention)
# so sample_order/incident_id aren't corrupted by integer/NaN coercion.
module3 = pd.read_csv(MODULE3_CSV, dtype=str, keep_default_na=False)

print(f"Loaded {len(module3)} rows, {len(module3.columns)} columns")
expected_n = 160
if len(module3) != expected_n:
    print(f"WARNING: expected {expected_n} rows, got {len(module3)}")

# Base sampled incidents (purely numeric sample_order) vs. extra
# multi-outcome rows (alphabetic suffix, e.g. "44b", "44c").
is_extra = module3["sample_order"].str.contains(r"[a-z]$", regex=True)
base = module3[~is_extra].copy()
extra = module3[is_extra].copy()

print(f"Base sampled incidents: {len(base)}")
print(f"Extra multi-outcome rows: {len(extra)}")

# Check 1: exactly 150 unique base incidents, no gaps in sample_order 1-150.
base_orders = sorted(base["sample_order"].astype(int))
expected_orders = list(range(1, 151))
check1_pass = (
    base["sample_order"].nunique() == 150
    and len(base) == 150
    and base_orders == expected_orders
)
print()
print(f"CHECK 1 (150 unique base incidents, sample_order 1-150 with no gaps): "
      f"{'PASS' if check1_pass else 'FAIL'}")
if not check1_pass:
    missing = sorted(set(expected_orders) - set(base_orders))
    dupes = base["sample_order"][base["sample_order"].duplicated()].tolist()
    print(f"  n unique = {base['sample_order'].nunique()}, n rows = {len(base)}")
    if missing:
        print(f"  missing sample_order values: {missing}")
    if dupes:
        print(f"  duplicated sample_order values: {dupes}")

# Check 2: no blank coder values (every row actually coded).
blank_coder = module3["coder"] == ""
check2_pass = not blank_coder.any()
print(f"CHECK 2 (no blank 'coder' values): {'PASS' if check2_pass else 'FAIL'}")
if not check2_pass:
    print(f"  {blank_coder.sum()} row(s) with blank coder:")
    print(module3.loc[blank_coder, ["sample_order", "incident_id", "title"]].to_string(index=False))

In [ ]:
# Step 2: Core descriptive statistics
from scipy.stats import norm

N_BASE = len(base)  # should be 150

# 1. Headline hit rate: base incidents with >=1 outcome_type != "none_found".
hit_mask = base["outcome_type"] != "none_found"
n_hits = int(hit_mask.sum())
hit_rate_pct = n_hits / N_BASE * 100
print(f"1. Headline hit rate: {n_hits} / {N_BASE} = {hit_rate_pct:.1f}%")

# 2. Outcome type breakdown -- ALL outcome-bearing rows (base + extra),
# since extra rows are additional outcome instances for multi-outcome
# incidents, not a separate population.
outcome_rows = module3[module3["outcome_type"] != "none_found"]
outcome_counts = outcome_rows["outcome_type"].value_counts()
print()
print("2. Outcome type breakdown (excluding none_found):")
print(outcome_counts.to_string())
print(f"   Total instances: {outcome_counts.sum()}")

# 3. 95% CI on the hit rate: Wald and Wilson.
p_hat = n_hits / N_BASE
z = norm.ppf(0.975)
wald_margin = z * np.sqrt(p_hat * (1 - p_hat) / N_BASE)
wald_lo, wald_hi = p_hat - wald_margin, p_hat + wald_margin

try:
    from statsmodels.stats.proportion import proportion_confint
    wilson_lo, wilson_hi = proportion_confint(n_hits, N_BASE, method="wilson")
except ImportError:
    denom = 1 + z**2 / N_BASE
    center = (p_hat + z**2 / (2 * N_BASE)) / denom
    margin = (z * np.sqrt(p_hat * (1 - p_hat) / N_BASE + z**2 / (4 * N_BASE**2))) / denom
    wilson_lo, wilson_hi = center - margin, center + margin

print()
print(f"3. 95% CI on hit rate ({hit_rate_pct:.1f}%, n={N_BASE}):")
print(f"   Wald:   [{wald_lo:.1%}, {wald_hi:.1%}]")
print(f"   Wilson: [{wilson_lo:.1%}, {wilson_hi:.1%}]")

# 4. Hit rate by MIT risk domain.
domain_summary = (
    base.assign(has_outcome=hit_mask)
    .groupby("mit_risk_domain")
    .agg(incidents_sampled=("sample_order", "count"), incidents_with_outcome=("has_outcome", "sum"))
    .reset_index()
)
domain_summary["hit_rate_pct"] = (
    domain_summary["incidents_with_outcome"] / domain_summary["incidents_sampled"] * 100
).round(1)
print()
print("4. Hit rate by MIT risk domain:")
print(domain_summary.to_string(index=False))

# 5. Line of business breakdown among outcome rows.
lob_counts = outcome_rows["line_of_business"].value_counts(dropna=False)
print()
print("5. Line of business breakdown (outcome rows only; blank = no clear assignment):")
print(lob_counts.to_string())

# 6. Disclosed amounts.
has_amount = module3["amount_usd"].str.strip() != ""
amounts = module3[has_amount].copy()
amounts["amount_usd_float"] = (
    amounts["amount_usd"].str.replace(r"[$,]", "", regex=True).astype(float)
)
amounts_sorted = amounts.sort_values("amount_usd_float")

print()
print(f"6. Disclosed amounts: n = {len(amounts_sorted)}")
if len(amounts_sorted):
    print(f"   Min: ${amounts_sorted['amount_usd_float'].min():,.0f}")
    print(f"   Max: ${amounts_sorted['amount_usd_float'].max():,.0f}")
print()
print(amounts_sorted[["incident_id", "title", "outcome_type", "amount_usd_float"]].to_string(index=False))

In [ ]:
# Step 3: Cross-check against the paper
print("Step 3: Cross-check against paper draft values (Section 4.3)")
print("=" * 70)

check_results = []
mismatched_labels = []


def check(label, computed, paper, tol=1e-9):
    """Print MATCH/MISMATCH for a single computed-vs-paper value.

    A mismatch is flagged explicitly rather than silently accepted --
    resolving which side is right is for a human, not this notebook.
    """
    ok = computed is not None and abs(computed - paper) <= tol
    line = f"  {label}: notebook = {computed}, paper = {paper}"
    if ok:
        print(line + "  [MATCH]")
    else:
        print(line + f"  [MISMATCH: paper states {paper}, notebook computes {computed}]")
    check_results.append(ok)
    if not ok:
        mismatched_labels.append(label)
    return ok


check("Headline hit rate (n)", n_hits, 33)
check("Headline hit rate (%)", round(hit_rate_pct, 1), 22.0)
check("Outcome type: litigation", int(outcome_counts.get("litigation", 0)), 14)
check("Outcome type: criminal_prosecution", int(outcome_counts.get("criminal_prosecution", 0)), 9)
check("Outcome type: fine", int(outcome_counts.get("fine", 0)), 8)
check("Outcome type: settlement", int(outcome_counts.get("settlement", 0)), 6)
check("Outcome type: regulatory_enforcement", int(outcome_counts.get("regulatory_enforcement", 0)), 6)
check("Outcome type: total instances", int(outcome_counts.sum()), 43)
# Wilson CI is cited in the paper as "approximately 16%-29%" -- compared
# to the nearest percentage point rather than requiring exact equality.
check("Wilson CI lower bound (pp)", round(wilson_lo * 100), 16, tol=1)
check("Wilson CI upper bound (pp)", round(wilson_hi * 100), 29, tol=1)
check("Disclosed amounts (n)", len(amounts_sorted), 17)
check(
    "Disclosed amounts (min $)",
    amounts_sorted["amount_usd_float"].min() if len(amounts_sorted) else None,
    1874,
)
check(
    "Disclosed amounts (max $)",
    amounts_sorted["amount_usd_float"].max() if len(amounts_sorted) else None,
    140_000_000,
)

# Domain hit rates: match the paper's short domain names to this dataset's
# full "N. <name>" mit_risk_domain labels (same MIT taxonomy as Module 1).
DOMAIN_PAPER = {
    "1. Discrimination and Toxicity": ("Discrimination", 2, 19),
    "2. Privacy & Security": ("Privacy", 4, 10),
    "3. Misinformation": ("Misinformation", 5, 22),
    "4. Malicious Actors & Misuse": ("Misuse", 11, 61),
    "5. Human-Computer Interaction": ("HCI", 2, 8),
    "6. Socioeconomic & Environmental Harms": ("Socioeconomic", 3, 7),
    "7. AI system safety, failures, and limitations": ("Safety", 6, 23),
}
domain_summary_idx = domain_summary.set_index("mit_risk_domain")
for full_label, (short, paper_hits, paper_total) in DOMAIN_PAPER.items():
    if full_label in domain_summary_idx.index:
        computed_hits = int(domain_summary_idx.loc[full_label, "incidents_with_outcome"])
        computed_total = int(domain_summary_idx.loc[full_label, "incidents_sampled"])
    else:
        computed_hits, computed_total = None, None
        print(f"  NOTE: domain label '{full_label}' not found in this dataset's mit_risk_domain values.")
    check(f"Domain hit rate: {short} (hits)", computed_hits, paper_hits)
    check(f"Domain hit rate: {short} (total)", computed_total, paper_total)

# Line of business (only the five categories the paper gives explicit
# counts for; the residual "no clear assignment"/"other" rows are reported
# in Step 2 #5 above without a specific paper target to check against).
LOB_PAPER = {
    "media_liability": 7, "cyber": 7, "E&O_professional": 6,
    "product_liability": 4, "GL": 3,
}
for lob, paper_n in LOB_PAPER.items():
    computed_n = int(lob_counts.get(lob, 0))
    check(f"Line of business: {lob}", computed_n, paper_n)

all_pass = all(check_results)
print()
if all_pass:
    print("All paper-stated numbers in Section 4.3 match this notebook's computation.")
else:
    print(f"{len(mismatched_labels)} mismatch(es) found: {mismatched_labels}")

In [ ]:
# Step 4: Save outputs
domain_out = domain_summary[["mit_risk_domain", "incidents_sampled", "incidents_with_outcome", "hit_rate_pct"]]
domain_out_path = os.path.join(BATCH_DIR, "module3_summary_by_domain.csv")
domain_out.to_csv(domain_out_path, index=False)

amounts_out = amounts_sorted[["incident_id", "title", "outcome_type", "amount_usd_float"]].rename(
    columns={"amount_usd_float": "amount_usd"}
)
amounts_out_path = os.path.join(BATCH_DIR, "module3_disclosed_amounts.csv")
amounts_out.to_csv(amounts_out_path, index=False)

print(f"Saved {domain_out_path}")
print(f"Saved {amounts_out_path}")
print()
if all_pass:
    print("SUMMARY: every number in the paper's Section 4.3 is traceable to this notebook and matches.")
else:
    print("SUMMARY: NOT all numbers match. Mismatches found in:")
    for label in mismatched_labels:
        print(f"  - {label}")